In [7]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def preprocess(train_file, test_file):
    train = np.load(file=train_file, allow_pickle=True)
    test = np.load(file=test_file, allow_pickle=True)

    X_train, y_train = train['x'], train['y']
    print(X_train.shape, X_train.dtype, y_train.shape, y_train.dtype)
    X_test, y_test = test['x'], test['y']
    print(X_test.shape, X_test.dtype, y_test.shape, y_test.dtype)

    print(len(np.unique(y_train)))

    X_train = X_train.astype(np.float32)
    X_test = X_test.astype(np.float32)
    print(X_train.dtype, X_test.dtype)

    mean = np.mean(a=X_train, axis=0)
    std  = np.std(a=X_train, axis=0)
    X_train_norm = (X_train - mean) / std
    X_test_norm  = (X_test  - mean) / std

    n = X_train.shape[0]
    K = (X_train_norm.T @ X_train_norm) / (n - 1)
    print(K)

    D, V    = np.linalg.eigh(K)
    indices = np.argsort(a=D, descending=True)
    D = D[indices]
    V = V[:, indices]

    feats = 256
    train_PCA = X_train_norm @ V[:, :feats]
    test_PCA  = X_test_norm  @ V[:, :feats]

    train_PCA_white = train_PCA / np.sqrt(D[:feats])
    test_PCA_white  = test_PCA / np.sqrt(D[:feats])

    X_train = train_PCA_white
    X_test  = test_PCA_white

    return (X_train, y_train), (X_test, y_test)

(X_train, y_train), (X_test, y_test) = preprocess("Numarul6-train/1_6_train.npz", "Numarul6-test/1_6_test.npz")

(28000, 1024) object (28000,) int64
(7000, 1024) object (7000,) int64
7
float32 float32
[[1.0000353  0.99131685 0.9683875  ... 0.56507903 0.5512423  0.54065716]
 [0.99131685 1.0000349  0.9896635  ... 0.57449776 0.5597892  0.54848397]
 [0.9683875  0.9896635  1.000037   ... 0.584872   0.56887656 0.55669683]
 ...
 [0.56507903 0.57449776 0.584872   ... 1.0000368  0.9848046  0.954257  ]
 [0.5512423  0.5597892  0.56887656 ... 0.9848046  1.0000379  0.9877055 ]
 [0.54065716 0.54848397 0.55669683 ... 0.954257   0.9877055  1.0000327 ]]


In [8]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.act = nn.ReLU()

        self.fc1   = nn.Linear(in_features=256, out_features=128)
        self.drop1 = nn.Dropout(p=0.3)
        self.fc2   = nn.Linear(in_features=128, out_features=64)
        self.drop2 = nn.Dropout(p=0.5)
        self.fc3   = nn.Linear(in_features=64, out_features=32)
        self.drop3 = nn.Dropout(p=0.3)
        self.fc4   = nn.Linear(in_features=32, out_features=7)

    def forward(self, x):
        x = self.drop1(self.act(self.fc1(x)))
        x = self.drop2(self.act(self.fc2(x)))
        x = self.drop3(self.act(self.fc3(x)))
        x = self.fc4(x)

        return x

model = Model()

In [9]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}"
    )

torch.save(model.state_dict(), "model1.pth")

Epoch 1/100 | Train Loss: 1.8871 | Train Acc: 21.24% | Test Loss: 1.6298 | Test Acc: 37.64%
Epoch 2/100 | Train Loss: 1.4793 | Train Acc: 44.69% | Test Loss: 1.2056 | Test Acc: 57.47%
Epoch 3/100 | Train Loss: 1.2171 | Train Acc: 56.88% | Test Loss: 1.0223 | Test Acc: 65.41%
Epoch 4/100 | Train Loss: 1.0812 | Train Acc: 62.95% | Test Loss: 0.9326 | Test Acc: 68.07%
Epoch 5/100 | Train Loss: 0.9898 | Train Acc: 66.38% | Test Loss: 0.8706 | Test Acc: 69.87%
Epoch 6/100 | Train Loss: 0.9381 | Train Acc: 68.42% | Test Loss: 0.8398 | Test Acc: 71.57%
Epoch 7/100 | Train Loss: 0.8951 | Train Acc: 69.97% | Test Loss: 0.8160 | Test Acc: 71.99%
Epoch 8/100 | Train Loss: 0.8584 | Train Acc: 71.13% | Test Loss: 0.7939 | Test Acc: 72.37%
Epoch 9/100 | Train Loss: 0.8290 | Train Acc: 72.33% | Test Loss: 0.7694 | Test Acc: 73.43%
Epoch 10/100 | Train Loss: 0.8047 | Train Acc: 73.31% | Test Loss: 0.7531 | Test Acc: 74.54%
Epoch 11/100 | Train Loss: 0.7756 | Train Acc: 74.37% | Test Loss: 0.7454 | Tes